# Phase 1: Face Detection with YOLOv8 + WIDER Face

**Professional Training - Final Project (Level 2)**

**Team Member:** Code Lead  
**Model:** YOLOv8 (trained from scratch on WIDER Face)  
**Dataset:** WIDER Face  
**Output:** Face detection with bounding boxes + confidence scores + metrics table

## Cell 1: Environment Setup

In [ ]:
# Install required packages (run once)
!pip install ultralytics opencv-python-headless matplotlib pandas seaborn -q

# Verify installation
print("\n" + "="*60)
print("Verifying package installations...")
print("="*60)

import numpy as np
print(f"✓ NumPy version: {np.__version__}")

import torch
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

import ultralytics
print(f"✓ Ultralytics version: {ultralytics.__version__}")

import cv2
print(f"✓ OpenCV version: {cv2.__version__}")

print("="*60)
print("All packages installed successfully!")
print("="*60)

## Cell 2: Imports

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO
from collections import defaultdict
import time

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

print("✓ All imports successful")
print(f"✓ Random seed set to: {RANDOM_SEED}")

## Cell 3: Create Directory Structure

In [ ]:
# Create directories
DATA_DIR = Path("data")
WIDER_DIR = DATA_DIR / "wider_face"
WIDER_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Create expected structure for WIDER Face
(WIDER_DIR / "WIDER_train" / "images").mkdir(parents=True, exist_ok=True)
(WIDER_DIR / "WIDER_val" / "images").mkdir(parents=True, exist_ok=True)
(WIDER_DIR / "wider_face_split").mkdir(parents=True, exist_ok=True)

print("✓ Directory structure created")
print(f"  Data directory: {WIDER_DIR.absolute()}")
print(f"  Results directory: {RESULTS_DIR.absolute()}")

## Cell 4: WIDER Face to YOLO Format Converter

**IMPORTANT:** You must download WIDER Face dataset first!

1. Go to: http://shuoyang1213.me/WIDERFACE/
2. Download: `WIDER_train.zip`, `WIDER_val.zip`, `wider_face_split.zip`
3. Extract to: `data/wider_face/`

In [ ]:
def convert_wider_to_yolo(wider_dir, split='train'):
    """
    Convert WIDER Face annotations to YOLO format.

    YOLO format per line: <class_id> <x_center> <y_center> <width> <height>
    All values are normalized to [0, 1]

    WIDER Face annotation format:
    - Line 1: image filename
    - Line 2: number of faces
    - Next N lines: x, y, w, h, blur, expression, illumination, invalid, occlusion, pose
    """
    wider_dir = Path(wider_dir)

    # Set paths based on split
    if split == 'train':
        img_dir = wider_dir / "WIDER_train" / "images"
        annot_file = wider_dir / "wider_face_split" / "wider_face_train_bbx_gt.txt"
    else:
        img_dir = wider_dir / "WIDER_val" / "images"
        annot_file = wider_dir / "wider_face_split" / "wider_face_val_bbx_gt.txt"

    output_dir = wider_dir / f"labels_{split}"
    output_dir.mkdir(exist_ok=True)

    # Check if annotation file exists
    if not annot_file.exists():
        print(f"❌ Annotation file not found: {annot_file}")
        print("   Please download WIDER Face dataset first!")
        print("   Visit: http://shuoyang1213.me/WIDERFACE/")
        return None

    print(f"\n📖 Reading annotations from: {annot_file}")
    with open(annot_file, 'r') as f:
        lines = f.readlines()

    idx = 0
    total_faces = 0
    total_images = 0
    skipped_images = 0

    while idx < len(lines):
        # Read image filename
        img_name = lines[idx].strip()
        idx += 1

        # Read number of faces
        num_faces = int(lines[idx].strip())
        idx += 1

        # Build image path
        img_path = img_dir / img_name

        if img_path.exists():
            # Read image to get dimensions
            img = cv2.imread(str(img_path))
            if img is not None:
                h, w = img.shape[:2]

                # Create label filename (replace / with _ to flatten structure)
                label_name = img_name.replace('.jpg', '.txt').replace('/', '_')
                label_path = output_dir / label_name

                # Write YOLO format annotations
                with open(label_path, 'w') as lf:
                    for _ in range(num_faces):
                        if idx >= len(lines):
                            break

                        bbox = lines[idx].strip().split()
                        # Parse: x, y, width, height (first 4 values)
                        x, y, bw, bh = map(int, bbox[:4])

                        # Convert to YOLO format (normalized)
                        x_center = (x + bw / 2.0) / w
                        y_center = (y + bh / 2.0) / h
                        width = bw / float(w)
                        height = bh / float(h)

                        # Clamp to [0, 1] to handle edge cases
                        x_center = max(0.0, min(1.0, x_center))
                        y_center = max(0.0, min(1.0, y_center))
                        width = max(0.0, min(1.0, width))
                        height = max(0.0, min(1.0, height))

                        # Write: class_id=0 (face), x_center, y_center, width, height
                        lf.write(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
                        total_faces += 1
                        idx += 1

                total_images += 1
            else:
                # Image file exists but cannot be read
                skipped_images += 1
                for _ in range(num_faces):
                    idx += 1
        else:
            # Image file does not exist - skip annotations
            skipped_images += 1
            for _ in range(num_faces):
                idx += 1

    print(f"\n✅ Conversion complete for '{split}' split:")
    print(f"   Images processed: {total_images}")
    print(f"   Total faces: {total_faces}")
    print(f"   Skipped (missing/unreadable): {skipped_images}")
    print(f"   Labels saved to: {output_dir}")

    return output_dir

### Run the conversion (after downloading dataset)

In [ ]:
print("="*60)
print("CONVERTING WIDER FACE TO YOLO FORMAT")
print("="*60)

# Convert training set
train_labels = convert_wider_to_yolo(WIDER_DIR, 'train')

# Convert validation set
val_labels = convert_wider_to_yolo(WIDER_DIR, 'val')

if train_labels is None or val_labels is None:
    print("\n⚠️  Conversion failed. Please ensure WIDER Face dataset is downloaded.")
    print("   Expected structure:")
    print("   data/wider_face/")
    print("   ├── WIDER_train/images/")
    print("   ├── WIDER_val/images/")
    print("   └── wider_face_split/")

## Cell 5: Create YOLO Dataset Configuration (YAML)

In [ ]:
def create_dataset_yaml(wider_dir, output_path="data/wider_face.yaml"):
    """
    Create YOLO dataset configuration file.
    YOLOv8 requires this YAML to know where images and labels are.
    """
    wider_dir = Path(wider_dir).absolute()

    # Verify paths exist
    train_path = wider_dir / "WIDER_train" / "images"
    val_path = wider_dir / "WIDER_val" / "images"

    print(f"\nChecking paths:")
    print(f"   Train: {train_path} -> {'✅ Exists' if train_path.exists() else '❌ Missing'}")
    print(f"   Val:   {val_path} -> {'✅ Exists' if val_path.exists() else '❌ Missing'}")

    yaml_content = f"""# WIDER Face Dataset Configuration for YOLOv8
# Auto-generated for Professional Training Final Project

path: {wider_dir}  # Dataset root directory

train: WIDER_train/images  # Train images (relative to 'path')
val: WIDER_val/images      # Validation images (relative to 'path')

# Class names
names:
  0: face

# Number of classes
nc: 1
"""

    with open(output_path, 'w') as f:
        f.write(yaml_content)

    print(f"\n✅ Dataset YAML created: {Path(output_path).absolute()}")
    return output_path

# Create the YAML file
yaml_path = create_dataset_yaml(WIDER_DIR)

# Display contents
print("\n📄 YAML Contents:")
with open(yaml_path, 'r') as f:
    print(f.read())

## Cell 6: Verify Data Before Training

In [ ]:
def verify_dataset(wider_dir):
    """Verify that dataset is ready for training."""
    wider_dir = Path(wider_dir)

    print("="*60)
    print("DATASET VERIFICATION")
    print("="*60)

    checks = {}

    # Check 1: Train images
    train_img_dir = wider_dir / "WIDER_train" / "images"
    train_images = list(train_img_dir.rglob("*.jpg")) if train_img_dir.exists() else []
    checks['train_images'] = len(train_images)
    print(f"\n1. Train images: {len(train_images)} {'✅' if len(train_images) > 0 else '❌'}")

    # Check 2: Val images
    val_img_dir = wider_dir / "WIDER_val" / "images"
    val_images = list(val_img_dir.rglob("*.jpg")) if val_img_dir.exists() else []
    checks['val_images'] = len(val_images)
    print(f"2. Val images: {len(val_images)} {'✅' if len(val_images) > 0 else '❌'}")

    # Check 3: Train labels
    train_labels_dir = wider_dir / "labels_train"
    train_labels = list(train_labels_dir.glob("*.txt")) if train_labels_dir.exists() else []
    checks['train_labels'] = len(train_labels)
    print(f"3. Train labels: {len(train_labels)} {'✅' if len(train_labels) > 0 else '❌'}")

    # Check 4: Val labels
    val_labels_dir = wider_dir / "labels_val"
    val_labels = list(val_labels_dir.glob("*.txt")) if val_labels_dir.exists() else []
    checks['val_labels'] = len(val_labels)
    print(f"4. Val labels: {len(val_labels)} {'✅' if len(val_labels) > 0 else '❌'}")

    # Check 5: YAML file
    yaml_file = Path("data/wider_face.yaml")
    checks['yaml'] = yaml_file.exists()
    print(f"5. YAML config: {'✅' if yaml_file.exists() else '❌'}")

    # Summary
    print(f"\n{'='*60}")
    all_good = all([
        checks['train_images'] > 0,
        checks['val_images'] > 0,
        checks['train_labels'] > 0,
        checks['val_labels'] > 0,
        checks['yaml']
    ])

    if all_good:
        print("✅ DATASET READY FOR TRAINING")
    else:
        print("❌ DATASET NOT READY - Please fix issues above")
    print("="*60)

    return all_good

# Run verification
is_ready = verify_dataset(WIDER_DIR)

## Cell 7: Train YOLOv8 Model

**⚠️ This cell takes time!**
- GPU: ~1-2 hours for 50 epochs
- CPU: ~8-12 hours for 50 epochs

If short on time, reduce `epochs` to 25 first.

In [ ]:
def train_face_detector(data_yaml, epochs=50, imgsz=640, batch=16):
    """
    Train YOLOv8n on WIDER Face dataset.

    Args:
        data_yaml: Path to dataset YAML file
        epochs: Number of training epochs
        imgsz: Input image size
        batch: Batch size (reduce if out of memory)

    Returns:
        model: Trained YOLO model
        results: Training results object
    """
    print("="*60)
    print("STARTING YOLOv8 FACE DETECTION TRAINING")
    print("="*60)
    print(f"📊 Configuration:")
    print(f"   Model: YOLOv8n (nano - fastest)")
    print(f"   Epochs: {epochs}")
    print(f"   Image size: {imgsz}")
    print(f"   Batch size: {batch}")
    print(f"   Device: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")
    print(f"   Dataset: {data_yaml}")
    print("="*60)
    print("\n⏳ Training in progress... (this will take time)\n")

    # Load pretrained YOLOv8n (COCO weights as starting point)
    model = YOLO('yolov8n.pt')

    # Train
    start_time = time.time()

    results = model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        name='face_detection',
        project='runs/detect',
        patience=10,           # Early stopping if no improvement for 10 epochs
        save=True,             # Save checkpoints
        device=0 if torch.cuda.is_available() else 'cpu',
        verbose=True,          # Print progress
        seed=RANDOM_SEED       # Reproducibility
    )

    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print("✅ TRAINING COMPLETE!")
    print(f"   Time elapsed: {elapsed/60:.1f} minutes")
    print(f"   Best model: runs/detect/face_detection/weights/best.pt")
    print(f"   Last model: runs/detect/face_detection/weights/last.pt")
    print("="*60)

    return model, results

# Run training (only if dataset is ready)
if is_ready:
    # Adjust batch size based on available memory
    if torch.cuda.is_available():
        # Try batch=16, fallback to 8 if needed
        try:
            model, results = train_face_detector('data/wider_face.yaml', epochs=50, batch=16)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("\n⚠️  GPU out of memory. Retrying with batch=8...")
                torch.cuda.empty_cache()
                model, results = train_face_detector('data/wider_face.yaml', epochs=50, batch=8)
            else:
                raise
    else:
        # CPU training - use smaller batch
        print("\n⚠️  No GPU detected. Training on CPU will be slow.")
        print("   Consider using Google Colab or Kaggle for free GPU access.\n")
        model, results = train_face_detector('data/wider_face.yaml', epochs=25, batch=4)
else:
    print("\n❌ Cannot start training - dataset not ready.")
    print("   Please download WIDER Face and run conversion first.")

## Cell 8: Check Training Results

In [ ]:
def check_training_results():
    """Check if training completed and display metrics."""
    results_path = Path("runs/detect/face_detection/results.csv")

    print("="*60)
    print("TRAINING RESULTS CHECK")
    print("="*60)

    if not results_path.exists():
        print("\n❌ No results.csv found.")
        print("   Possible causes:")
        print("   - Training is still running")
        print("   - Training failed (check error messages above)")
        print("   - Training was interrupted")

        # Check if any runs exist
        runs_dir = Path("runs/detect")
        if runs_dir.exists():
            subdirs = [d for d in runs_dir.iterdir() if d.is_dir()]
            if subdirs:
                print(f"\n   Found run directories: {[d.name for d in subdirs]}")
        return None

    # Load results
    results_df = pd.read_csv(results_path)
    print(f"\n✅ Training results found!")
    print(f"   Total epochs completed: {len(results_df)}")
    print(f"\nLast 5 epochs:")
    display(results_df.tail())

    # Plot metrics
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # mAP@0.5
    if 'metrics/mAP50(B)' in results_df.columns:
        axes[0,0].plot(results_df['epoch'], results_df['metrics/mAP50(B)'], 'b-o', markersize=4)
        axes[0,0].set_title('mAP@0.5 (IoU=0.5)', fontsize=12, fontweight='bold')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('mAP')
        axes[0,0].grid(alpha=0.3)
        axes[0,0].set_ylim(0, 1)

    # mAP@0.5:0.95
    if 'metrics/mAP50-95(B)' in results_df.columns:
        axes[0,1].plot(results_df['epoch'], results_df['metrics/mAP50-95(B)'], 'g-o', markersize=4)
        axes[0,1].set_title('mAP@0.5:0.95 (COCO metric)', fontsize=12, fontweight='bold')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('mAP')
        axes[0,1].grid(alpha=0.3)
        axes[0,1].set_ylim(0, 1)

    # Box loss
    if 'train/box_loss' in results_df.columns:
        axes[1,0].plot(results_df['epoch'], results_df['train/box_loss'], 'r-', label='Train', linewidth=2)
        if 'val/box_loss' in results_df.columns:
            axes[1,0].plot(results_df['epoch'], results_df['val/box_loss'], 'orange', label='Val', linewidth=2)
        axes[1,0].set_title('Box Regression Loss', fontsize=12, fontweight='bold')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Loss')
        axes[1,0].legend()
        axes[1,0].grid(alpha=0.3)

    # Classification loss
    if 'train/cls_loss' in results_df.columns:
        axes[1,1].plot(results_df['epoch'], results_df['train/cls_loss'], 'r-', label='Train', linewidth=2)
        if 'val/cls_loss' in results_df.columns:
            axes[1,1].plot(results_df['epoch'], results_df['val/cls_loss'], 'orange', label='Val', linewidth=2)
        axes[1,1].set_title('Classification Loss', fontsize=12, fontweight='bold')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Loss')
        axes[1,1].legend()
        axes[1,1].grid(alpha=0.3)

    plt.suptitle('YOLOv8 Face Detection Training Metrics', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'training_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Final metrics summary
    final = results_df.iloc[-1]
    print(f"\n{'='*60}")
    print("FINAL TRAINING METRICS")
    print(f"{'='*60}")

    metrics_display = []
    if 'metrics/mAP50(B)' in final:
        metrics_display.append(('mAP@0.5', final['metrics/mAP50(B)']))
    if 'metrics/mAP50-95(B)' in final:
        metrics_display.append(('mAP@0.5:0.95', final['metrics/mAP50-95(B)']))
    if 'metrics/precision(B)' in final:
        metrics_display.append(('Precision', final['metrics/precision(B)']))
    if 'metrics/recall(B)' in final:
        metrics_display.append(('Recall', final['metrics/recall(B)']))

    for name, value in metrics_display:
        status = "✅ Good" if value > 0.75 else ("⚠️  Acceptable" if value > 0.60 else "❌ Needs work")
        print(f"   {name:15s}: {value:.4f}  {status}")

    print(f"{'='*60}")

    # Save metrics to CSV
    metrics_df = pd.DataFrame(metrics_display, columns=['Metric', 'Value'])
    metrics_df.to_csv(RESULTS_DIR / 'face_detection_metrics.csv', index=False)
    print(f"\n✅ Metrics saved to: {RESULTS_DIR / 'face_detection_metrics.csv'}")
    print(f"✅ Plot saved to: {RESULTS_DIR / 'training_metrics.png'}")

    return results_df

# Run check
results_df = check_training_results()

## Cell 9: Face Detector Class (Inference)

In [ ]:
class FaceDetector:
    """
    Face Detection wrapper using trained YOLOv8 model.

    Usage:
        detector = FaceDetector('runs/detect/face_detection/weights/best.pt')
        detections = detector.detect('image.jpg')
    """

    def __init__(self, model_path='runs/detect/face_detection/weights/best.pt', conf_threshold=0.5):
        """
        Initialize detector.

        Args:
            model_path: Path to trained YOLO weights
            conf_threshold: Minimum confidence to keep detection
        """
        self.model_path = model_path
        self.conf_threshold = conf_threshold

        # Load model
        if Path(model_path).exists():
            self.model = YOLO(model_path)
            print(f"✅ Loaded trained model: {model_path}")
        else:
            # Fallback to pretrained
            print(f"⚠️  Trained model not found at {model_path}")
            print("   Falling back to pretrained YOLOv8n (COCO)")
            self.model = YOLO('yolov8n.pt')

        self.class_names = ['face']

    def detect(self, image_path, conf_threshold=None):
        """
        Detect faces in an image.

        Args:
            image_path: Path to image file
            conf_threshold: Override default confidence threshold

        Returns:
            List of detection dicts with keys: bbox, confidence, class
        """
        conf = conf_threshold if conf_threshold is not None else self.conf_threshold

        # Run inference
        results = self.model(image_path, conf=conf, verbose=False)

        detections = []
        for result in results:
            boxes = result.boxes
            if boxes is not None:
                for box in boxes:
                    # Get bounding box coordinates
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    conf_score = float(box.conf[0].cpu().numpy())
                    cls_id = int(box.cls[0].cpu().numpy())

                    detections.append({
                        'bbox': [int(x1), int(y1), int(x2), int(y2)],
                        'confidence': conf_score,
                        'class': self.class_names[cls_id] if cls_id < len(self.class_names) else 'face'
                    })

        return detections

    def draw_detections(self, image_path, detections, save_path=None):
        """
        Draw bounding boxes and labels on image.

        Args:
            image_path: Original image path
            detections: List of detection dicts from detect()
            save_path: Optional path to save annotated image

        Returns:
            Annotated image as RGB numpy array
        """
        img = cv2.imread(str(image_path))
        if img is None:
            raise ValueError(f"Could not read image: {image_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        for det in detections:
            x1, y1, x2, y2 = det['bbox']
            conf = det['confidence']

            # Draw bounding box (green)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Draw label background
            label = f"Face {conf:.2f}"
            (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
            cv2.rectangle(img, (x1, y1 - text_h - 10), (x1 + text_w, y1), (0, 255, 0), -1)

            # Draw label text
            cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)

        if save_path:
            cv2.imwrite(str(save_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

        return img

# Initialize detector (will use trained model if available)
detector = FaceDetector()

## Cell 10: Test on Sample Images

In [ ]:
def test_on_validation_sample(num_samples=5):
    """Test detector on random validation images."""
    val_dir = Path("data/wider_face/WIDER_val/images")

    if not val_dir.exists():
        print("❌ Validation images not found")
        return

    # Get random sample of images
    all_images = list(val_dir.rglob("*.jpg"))
    if len(all_images) == 0:
        print("❌ No images found in validation directory")
        return

    sample_images = np.random.choice(all_images, min(num_samples, len(all_images)), replace=False)

    print(f"\n🖼️  Testing on {len(sample_images)} random validation images...\n")

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    all_detections = []

    for idx, img_path in enumerate(sample_images):
        # Detect
        detections = detector.detect(str(img_path))
        all_detections.extend(detections)

        # Draw
        annotated = detector.draw_detections(img_path, detections)

        axes[idx].imshow(annotated)
        axes[idx].set_title(f"{img_path.name}\n{len(detections)} face(s)", fontsize=10)
        axes[idx].axis('off')

    # Hide unused subplots
    for idx in range(len(sample_images), 6):
        axes[idx].axis('off')

    plt.suptitle('Face Detection Results on Validation Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'validation_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n✅ Results saved to: {RESULTS_DIR / 'validation_samples.png'}")
    return all_detections

# Run test
val_detections = test_on_validation_sample(num_samples=5)

## Cell 11: Compute Detection Metrics

In [ ]:
def compute_detection_metrics(detections_list, save_results=True):
    """
    Compute and visualize detection metrics.

    Args:
        detections_list: List of detection dicts
        save_results: Whether to save results to files

    Returns:
        metrics dict, metrics DataFrame
    """
    if not detections_list:
        print("⚠️  No detections provided")
        return None, None

    confidences = [d['confidence'] for d in detections_list]

    metrics = {
        'total_detections': len(detections_list),
        'avg_confidence': np.mean(confidences),
        'min_confidence': np.min(confidences),
        'max_confidence': np.max(confidences),
        'std_confidence': np.std(confidences),
        'median_confidence': np.median(confidences)
    }

    print("="*60)
    print("DETECTION METRICS")
    print("="*60)
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"   {key:20s}: {value:.4f}")
        else:
            print(f"   {key:20s}: {value}")
    print("="*60)

    # Visualizations
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # 1. Detection count per image (if we have image info)
    # For now, just show total
    axes[0].bar(['Total Faces\nDetected'], [metrics['total_detections']], 
                color='steelblue', edgecolor='navy', linewidth=2)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Total Detections', fontsize=12, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    for i, v in enumerate([metrics['total_detections']]):
        axes[0].text(i, v + 0.5, str(v), ha='center', fontsize=14, fontweight='bold')

    # 2. Confidence distribution histogram
    axes[1].hist(confidences, bins=20, color='lightgreen', edgecolor='darkgreen', 
                 alpha=0.7, linewidth=1.5)
    axes[1].axvline(metrics['avg_confidence'], color='red', linestyle='--', 
                    linewidth=2, label=f'Mean: {metrics["avg_confidence"]:.3f}')
    axes[1].axvline(metrics['median_confidence'], color='blue', linestyle=':', 
                    linewidth=2, label=f'Median: {metrics["median_confidence"]:.3f}')
    axes[1].set_xlabel('Confidence Score', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('Confidence Distribution', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)

    # 3. Confidence statistics bar chart
    stats_names = ['Min', 'Max', 'Mean', 'Median', 'Std']
    stats_values = [
        metrics['min_confidence'],
        metrics['max_confidence'],
        metrics['avg_confidence'],
        metrics['median_confidence'],
        metrics['std_confidence']
    ]
    colors = ['#ff6b6b', '#51cf66', '#339af0', '#fcc419', '#9775fa']
    bars = axes[2].bar(stats_names, stats_values, color=colors, edgecolor='black', linewidth=1.5)
    axes[2].set_ylabel('Confidence Score', fontsize=12)
    axes[2].set_title('Confidence Statistics', fontsize=12, fontweight='bold')
    axes[2].set_ylim(0, 1.1)
    axes[2].grid(axis='y', alpha=0.3)

    # Add value labels on bars
    for bar, val in zip(bars, stats_values):
        height = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.suptitle('Face Detection Performance Metrics', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()

    if save_results:
        plt.savefig(RESULTS_DIR / 'detection_metrics.png', dpi=150, bbox_inches='tight')
        print(f"\n✅ Metrics plot saved to: {RESULTS_DIR / 'detection_metrics.png'}")

    plt.show()

    # Create metrics DataFrame
    metrics_df = pd.DataFrame([
        {'Metric': 'Total Faces Detected', 'Value': metrics['total_detections']},
        {'Metric': 'Average Confidence', 'Value': f"{metrics['avg_confidence']:.4f}"},
        {'Metric': 'Minimum Confidence', 'Value': f"{metrics['min_confidence']:.4f}"},
        {'Metric': 'Maximum Confidence', 'Value': f"{metrics['max_confidence']:.4f}"},
        {'Metric': 'Median Confidence', 'Value': f"{metrics['median_confidence']:.4f}"},
        {'Metric': 'Std Confidence', 'Value': f"{metrics['std_confidence']:.4f}"}
    ])

    if save_results:
        metrics_df.to_csv(RESULTS_DIR / 'detection_metrics_table.csv', index=False)
        print(f"✅ Metrics table saved to: {RESULTS_DIR / 'detection_metrics_table.csv'}")

    display(metrics_df)

    return metrics, metrics_df

# Compute metrics from validation detections
if val_detections:
    metrics, metrics_df = compute_detection_metrics(val_detections)

## Cell 12: Batch Processing

In [ ]:
def batch_detect(image_folder, output_folder="results/batch_output", conf_threshold=0.5):
    """
    Process multiple images and save results.

    Args:
        image_folder: Directory containing images
        output_folder: Directory to save annotated images
        conf_threshold: Confidence threshold

    Returns:
        List of all detection results
    """
    image_folder = Path(image_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    # Find all images
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        image_files.extend(image_folder.glob(ext))
        image_files.extend(image_folder.glob(ext.upper()))

    print(f"\n📁 Found {len(image_files)} images in {image_folder}")

    if len(image_files) == 0:
        print("❌ No images found")
        return []

    all_results = []
    detection_counts = []

    print(f"\n⏳ Processing images...\n")

    for idx, img_path in enumerate(image_files, 1):
        # Detect faces
        detections = detector.detect(str(img_path), conf_threshold=conf_threshold)
        detection_counts.append(len(detections))

        # Save annotated image
        if len(detections) > 0:
            annotated = detector.draw_detections(img_path, detections)
            save_path = output_folder / f"detected_{img_path.name}"
            cv2.imwrite(str(save_path), cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

        # Record results
        for det in detections:
            all_results.append({
                'image': img_path.name,
                'bbox_x1': det['bbox'][0],
                'bbox_y1': det['bbox'][1],
                'bbox_x2': det['bbox'][2],
                'bbox_y2': det['bbox'][3],
                'confidence': det['confidence'],
                'class': det['class']
            })

        if idx % 10 == 0 or idx == len(image_files):
            print(f"   [{idx}/{len(image_files)}] {img_path.name}: {len(detections)} faces")

    # Save results CSV
    if all_results:
        results_df = pd.DataFrame(all_results)
        csv_path = output_folder / "detection_results.csv"
        results_df.to_csv(csv_path, index=False)
        print(f"\n✅ Results saved to: {csv_path}")
        print(f"   Total faces detected: {len(all_results)}")
        print(f"   Images with faces: {sum(1 for c in detection_counts if c > 0)}/{len(image_files)}")
    else:
        print("\n⚠️  No faces detected in any image")

    return all_results

# Example usage (uncomment to run):
# batch_results = batch_detect("test_images/", "results/batch_output")

## Cell 13: Model Validation (Official mAP)

In [ ]:
def validate_model(data_yaml='data/wider_face.yaml'):
    """
    Run official YOLO validation to compute mAP metrics.
    This is the proper way to evaluate object detection performance.
    """
    best_model_path = "runs/detect/face_detection/weights/best.pt"

    if not Path(best_model_path).exists():
        print(f"❌ Trained model not found: {best_model_path}")
        print("   Please complete training first (Cell 7)")
        return None

    print("="*60)
    print("RUNNING OFFICIAL MODEL VALIDATION")
    print("="*60)
    print(f"   Model: {best_model_path}")
    print(f"   Dataset: {data_yaml}")
    print("\n⏳ Validating... (this may take a few minutes)\n")

    # Load best model
    model = YOLO(best_model_path)

    # Run validation
    metrics = model.val(data=data_yaml, split='val', verbose=True)

    print(f"\n{'='*60}")
    print("VALIDATION RESULTS")
    print(f"{'='*60}")

    # Extract metrics
    val_results = {
        'mAP@0.5': metrics.box.map50,
        'mAP@0.5:0.95': metrics.box.map,
        'Precision': metrics.box.mp,
        'Recall': metrics.box.mr,
        'F1-Score': 2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr + 1e-16)
    }

    for name, value in val_results.items():
        status = "✅" if value > 0.75 else ("⚠️ " if value > 0.60 else "❌")
        print(f"   {name:15s}: {value:.4f}  {status}")

    print(f"{'='*60}")

    # Save validation results
    val_df = pd.DataFrame(list(val_results.items()), columns=['Metric', 'Value'])
    val_df.to_csv(RESULTS_DIR / 'validation_metrics.csv', index=False)
    print(f"\n✅ Validation metrics saved to: {RESULTS_DIR / 'validation_metrics.csv'}")

    return val_results

# Run validation (only if model is trained)
if Path("runs/detect/face_detection/weights/best.pt").exists():
    val_results = validate_model()
else:
    print("⚠️  Skipping validation - no trained model found")
    print("   Train the model first using Cell 7")

## Cell 14: Test on Custom Image

In [ ]:
def test_single_image(image_path, conf_threshold=0.3, save_result=True):
    """
    Test face detection on a single image with full visualization.

    Args:
        image_path: Path to test image
        conf_threshold: Detection confidence threshold
        save_result: Whether to save annotated image

    Returns:
        detections list, metrics DataFrame
    """
    image_path = Path(image_path)

    if not image_path.exists():
        print(f"❌ Image not found: {image_path}")
        print("   Please provide a valid image path")
        return None, None

    print(f"\n🖼️  Testing on: {image_path.name}")
    print(f"   Confidence threshold: {conf_threshold}")

    # Detect
    start_time = time.time()
    detections = detector.detect(str(image_path), conf_threshold=conf_threshold)
    inference_time = time.time() - start_time

    print(f"\n   ⏱️  Inference time: {inference_time*1000:.1f} ms")
    print(f"   👤 Faces detected: {len(detections)}")

    for i, det in enumerate(detections):
        print(f"      Face {i+1}: conf={det['confidence']:.3f}, "
              f"bbox=[{det['bbox'][0]}, {det['bbox'][1]}, {det['bbox'][2]}, {det['bbox'][3]}]")

    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Original image
    original = cv2.imread(str(image_path))
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    axes[0].imshow(original)
    axes[0].set_title('Original Image', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # Annotated image
    if len(detections) > 0:
        annotated = detector.draw_detections(image_path, detections)
        axes[1].imshow(annotated)
        title = f'Detected {len(detections)} Face(s)\n({inference_time*1000:.1f} ms)'
    else:
        axes[1].imshow(original)
        title = 'No Faces Detected'

    axes[1].set_title(title, fontsize=12, fontweight='bold')
    axes[1].axis('off')

    plt.suptitle(f'Face Detection: {image_path.name}', fontsize=14, fontweight='bold')
    plt.tight_layout()

    if save_result:
        save_path = RESULTS_DIR / f"test_{image_path.stem}.png"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n✅ Result saved to: {save_path}")

    plt.show()

    # Create results table
    if detections:
        results_table = pd.DataFrame([
            {
                'Face #': i+1,
                'Confidence': f"{d['confidence']:.4f}",
                'X1': d['bbox'][0],
                'Y1': d['bbox'][1],
                'X2': d['bbox'][2],
                'Y2': d['bbox'][3],
                'Width': d['bbox'][2] - d['bbox'][0],
                'Height': d['bbox'][3] - d['bbox'][1]
            }
            for i, d in enumerate(detections)
        ])
        display(results_table)
        return detections, results_table

    return detections, None

# Example: Test on a validation image
val_dir = Path("data/wider_face/WIDER_val/images")
if val_dir.exists():
    val_images = list(val_dir.rglob("*.jpg"))
    if val_images:
        # Pick a random image with faces (try a few)
        for _ in range(10):
            test_img = np.random.choice(val_images)
            detections, table = test_single_image(test_img, conf_threshold=0.3)
            if detections and len(detections) > 0:
                break

## Cell 15: Export Model for Deployment

In [ ]:
def export_model():
    """
    Export trained model to ONNX format for faster inference.
    This is useful for the Streamlit app later.
    """
    best_model_path = "runs/detect/face_detection/weights/best.pt"

    if not Path(best_model_path).exists():
        print("❌ No trained model to export")
        return

    print("="*60)
    print("EXPORTING MODEL")
    print("="*60)

    model = YOLO(best_model_path)

    # Export to ONNX
    print("\n⏳ Exporting to ONNX format...")
    model.export(format='onnx', imgsz=640, dynamic=True)

    # Export to TorchScript
    print("⏳ Exporting to TorchScript format...")
    model.export(format='torchscript', imgsz=640)

    print(f"\n✅ Export complete!")
    print(f"   ONNX: runs/detect/face_detection/weights/best.onnx")
    print(f"   TorchScript: runs/detect/face_detection/weights/best.torchscript")

# Uncomment to export:
# export_model()

## Cell 16: Summary & Next Steps

In [ ]:
def print_summary():
    """Print project summary and status."""
    print("="*70)
    print("  PHASE 1: FACE DETECTION - PROJECT SUMMARY")
    print("="*70)

    # Check what's been completed
    checks = {
        'Dataset downloaded': Path("data/wider_face/WIDER_train/images").exists(),
        'Labels converted': Path("data/wider_face/labels_train").exists(),
        'YAML created': Path("data/wider_face.yaml").exists(),
        'Model trained': Path("runs/detect/face_detection/weights/best.pt").exists(),
        'Results plotted': Path("results/training_metrics.png").exists(),
        'Validation tested': Path("results/validation_metrics.csv").exists()
    }

    print("\n📋 Completion Status:")
    for task, done in checks.items():
        status = "✅ Complete" if done else "❌ Not done"
        print(f"   {task:25s} {status}")

    print("\n📁 Generated Files:")
    results_dir = Path("results")
    if results_dir.exists():
        for f in sorted(results_dir.glob("*")):
            size = f.stat().st_size / 1024
            unit = "KB" if size < 1024 else "MB"
            size = size if size < 1024 else size / 1024
            print(f"   {f.name:40s} {size:.1f} {unit}")

    print("\n🚀 Next Steps (Phase 2):")
    print("   1. Download FER2013 dataset for expression recognition")
    print("   2. Train expression classification model (CNN)")
    print("   3. Map expressions to sentiment categories")
    print("   4. Integrate with face detection pipeline")

    print("\n" + "="*70)

print_summary()